# Regresión logística

A partir de lo observado en el análisis descritivo, las variables balls, strikes, plate_x, plate_z, sz_top, sz_bot parecen estar relacionadas con la variable creada Swing. 

Construimos y ajustamos dos modelos de regresión logística, uno con todas las variables disponibles, y otra incluyendo solo aquellas que en el análisis descriptivo mostraron indicios de estar relacionadas con la probabilidad de que un bateador haga *swing* ante un determinado pitcheo. Luego, comparamos ambos modelos mediante métricas de validación.

In [1]:
import polars as pl
import numpy as np
import pyprojroot
# ACORDARNOS DE AGREGAR A LAS DEPENDENCIAS pyarrow y la que no estaba en el tp1
ROOT = pyprojroot.here()

In [2]:
datos_entrenamiento = pl.read_parquet(ROOT / "Datos/datos_entrenamiento.parquet")
datos_validacion = pl.read_parquet(ROOT / "Datos/datos_validacion.parquet")

## Modelo completo

En primer lugar, realizamos un preprocesamiento de los datos. Para esto, estandarizamos las variables numéricas, a excepción de la velocidad del tiro, que previo a la estandarización consideramos su logaritmo para unificar su forma. Las variables categóricas las codificamos mediante *dummies*.

In [3]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn import set_config

from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression

num_attribs = [
    "plate_x",
    "plate_z",
    "sz_top",
    "sz_bot",
    "pfx_x",
    "pfx_z"   
]

asim_attribs = [
    "release_speed"
]

cat_attribs = [
    "balls",
    "strikes",
    "stand",
    "p_throws",
    "Lanzamiento"
]


num_pipeline = make_pipeline(StandardScaler())

log_pipeline = make_pipeline(
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler(),
)

cat_pipeline = make_pipeline(
    OneHotEncoder(handle_unknown="ignore", drop="first")
)

preprocessing = ColumnTransformer(
    [
        ("num", num_pipeline, num_attribs),
        ("log", log_pipeline, asim_attribs),
        ("cat", cat_pipeline, cat_attribs),
    ]
)

set_config(display="diagram")
preprocessing

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('log', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_n

Ajustamos el modelo de regresión logística usando todas las variables disponibles.

In [ ]:
vars = [
    "balls",
    "strikes",
    "stand",
    "p_throws",
    "Lanzamiento",
    "plate_x",
    "plate_z",
    "sz_top",
    "sz_bot",
    "release_speed",
    "pfx_x",
    "pfx_z"     
]

# Separamos la variable objetivo
x_train = datos_entrenamiento.select(vars).to_pandas()
y_train = datos_entrenamiento["Swing"].to_numpy()
x_test = datos_validacion.select(vars).to_pandas()
y_test = datos_validacion["Swing"].to_numpy()

# Agregamos el ajuste del modelo en el pipeline
model = make_pipeline(
    preprocessing,
    LogisticRegression(
        max_iter = 1000,
        random_state = 123        
    )
)

model.fit(x_train, y_train)

prob_test = model.predict_proba(x_test)[:, 1]

## Modelo reducido

Como se mencionó anteriormente, consideramos la construcción de un nuevo modelo de regresión logística usando aquellas variables que en análisis descriptivo presentaron algún indicio de relación con la variable respuesta. Previo al ajuste realizamos el preprocesamiento de los datos.

In [5]:
num_attribs2 = [
    "plate_x",
    "plate_z",
    "sz_top",
    "sz_bot"  
]

cat_attribs2 = [
    "balls",
    "strikes"
]

preprocessing2 = ColumnTransformer(
    [
        ("num", num_pipeline, num_attribs2),
        ("cat", cat_pipeline, cat_attribs2),
    ]
)

set_config(display="diagram")
preprocessing2

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

Ajustamos el modelo de regresión logística.

In [ ]:
vars2 = [
    "balls",
    "strikes",
    "plate_x",
    "plate_z",
    "sz_top",
    "sz_bot"
]

# Separamos la variable objetivo
x_train2 = datos_entrenamiento.select(vars2).to_pandas()
y_train2 = datos_entrenamiento["Swing"].to_numpy()
x_test2 = datos_validacion.select(vars2).to_pandas()
y_test2 = datos_validacion["Swing"].to_numpy()

# Agregamos el ajuste del modelo en el pipeline
model2 = make_pipeline(
    preprocessing2,
    LogisticRegression(
        max_iter = 1000,
        random_state = 123        
    )
)

model2.fit(x_train2, y_train2)

prob_test2 = model2.predict_proba(x_test2)[:, 1]

## Comparación de los modelos

Calculamos el log loss de ambos modelos y analizamos cuál ajustó mejor.

In [7]:
from sklearn.metrics import log_loss

log_loss_m = log_loss(y_test, prob_test)
log_loss_m2 = log_loss(y_test2, prob_test2)

print(f"Modelo 1: {log_loss_m} \nModelo 2: {log_loss_m2}")

Modelo 1: 0.5731238340359849 
Modelo 2: 0.5737904911927104


El *log-loss* del modelo completo (model) es un poco menor que para el que incluye solo algunas variables, lo que indica un mejor ajuste.

Calculamos métricas adicionales para elegir uno de los modelos propuestos.

In [8]:
from sklearn.metrics import log_loss, roc_auc_score, average_precision_score, brier_score_loss

# Modelo completo
log_loss_m = log_loss(y_test, prob_test)
roc_m = roc_auc_score(y_test, prob_test)
precision_m = average_precision_score(y_test, prob_test)
brier_m = brier_score_loss(y_test, prob_test)

# Modelo reducido
log_loss_m2 = log_loss(y_test2, prob_test2)
roc_m2 = roc_auc_score(y_test2, prob_test2)
precision_m2 = average_precision_score(y_test2, prob_test2)
brier_m2 = brier_score_loss(y_test2, prob_test2)

metricas = pl.DataFrame({
    "Modelo": ["Completo", "Reducido"],
    "Log Loss": [
        log_loss_m,
        log_loss_m2
    ],
    "ROC AUC": [
        roc_m,
        roc_m2
    ],
    "Average Precision": [
        precision_m,
        precision_m2
    ],
    "Brier Score": [
        brier_m,
        brier_m2
    ]
})

metricas

Modelo,Log Loss,ROC AUC,Average Precision,Brier Score
str,f64,f64,f64,f64
"""Completo""",0.573124,0.768499,0.624047,0.192772
"""Reducido""",0.57379,0.767849,0.623115,0.193027


Comparamos un modelo reducido, compuesto por las variables seleccionadas en el análisis descriptivo, con un modelo completo que incorporó la totalidad de las variables disponibles. El modelo completo presentó un desempeño levemente superior en todas las métricas evaluadas (*Log-Loss* = 0.5731, AUC = 0.7685, *Average Precision* = 0.6240 y Brier *Score* = 0.1928). Si bien las diferencias fueron pequeñas, elegimos el modelo completo dado que mejora la capacidad predictiva sin incrementar la complejidad del modelo.